# Calibrate Cost Basecase

Additional calibration of renewable cost to achieve zero investment in base case. Given price from the base case, we scale the marginal cost intercept to achieve zero investments

jab, cst 04.09.2019

## Packages and options

In [1]:
import pandas as pd
import gdxtools 
import matplotlib as mpl
import matplotlib.pyplot as plt
import glob
import os
import gams
import numpy as np
import gdxtools as gt
from math import ceil
%matplotlib inline

### Directories

In [2]:
fn_base = "../source_data/basecase.gdx" # benchmark data to get prices
fn_cost = "../../model/data/data_EU_2017_all.gdx" # standarad input file for remaining data
fn_out = "../../data_preparation/cost_parameters.xlsx"

### Display options

In [3]:
#Display very small numbers as zero
pd.set_option('display.chop_threshold', 0.000001)

In [4]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [5]:
mpl.rcParams["figure.figsize"] = (40,20)
mpl.rcParams["font.size"] = 40

## Intitialize gdx files

In [6]:
dir_gms = os.getcwd()
ws = gams.GamsWorkspace(dir_gms)

In [7]:
gdx_base = ws.add_database_from_gdx(fn_base)
gdx_cost = ws.add_database_from_gdx(fn_cost)

## Load data

### Prices

In [8]:
df_price = gt.get_symbol_values(gdx_base, "r_price", col_names=["country", "period"], kind="value")
df_price.head(1)

country  period
AT       t0001     38.566214
Name: Value, dtype: float64

### Profiles

In [9]:
df_beta = gt.get_symbol_values(gdx_base, "betaRen", col_names=["technology", "country", "period"], kind="value").to_frame("Value")
df_beta.head(1)

,,,Value
technology,country,period,
Solar,AT,t0008,0.000119


### Investment cost

In [10]:
df_cost_a = gt.get_symbol_values(gdx_cost, "cinv_0", col_names=["technology", "country"], kind="value").to_frame("a")
df_cost_b = gt.get_symbol_values(gdx_cost, "cinv_1", col_names=["technology", "country"], kind="value").to_frame("b")
df_cost = df_cost_a.join(df_cost_b)
df_cost.head(1)

,,a,b
technology,country,,
Solar,AT,36.213953,0.000000e+00


### Renewable production

In [11]:
df_ren = gt.get_symbol_values(gdx_base, "renTotal", col_names=["technology", "country"], kind="value").to_frame("generation")
df_ren.head(1)

,,generation
technology,country,
Solar,AT,1144304.365


## Unit profit

In [12]:
df_profit = df_price.reset_index().merge(df_beta.reset_index(), on=["country", "period"], how="right")
df_profit["revenue"] = df_profit.Value_x*df_profit.Value_y
df_profit = df_profit.groupby(["technology", "country"], as_index=False).revenue.sum()
df_profit.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 48 entries, 0 to 47
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   technology  48 non-null     object 
 1   country     48 non-null     object 
 2   revenue     48 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.5+ KB


In [13]:
df_profit.sort_values("country").head(1)

,technology,country,revenue
0,Solar,AT,36.300167


## Calculate new cost intercept

In [14]:
df_all = df_profit.set_index(["technology", "country"]).join(df_ren).join(df_cost).fillna(0)
df_all["a_new"] = df_all.revenue - df_all.generation*df_all.b
df_all

revenue    generation          a             b  \
technology   country                                                     
Solar        AT       36.300167  1.144304e+06  36.213953  0.000000e+00   
             BE       41.545160  5.776014e+06  42.156420  0.000000e+00   
             BG       40.877746  2.652000e+06  32.704350  0.000000e+00   
             CH       42.018984  4.209379e+05  34.595313  0.000000e+00   
             CZ       35.824808  4.326927e+06  37.605184  0.000000e+00   
             DE       35.589889  7.979000e+07  38.026617  0.000000e+00   
             DK       35.902495  1.577562e+06  42.478034  0.000000e+00   
             ES       52.134093  2.776462e+07  27.676798  0.000000e+00   
             FR       44.403209  1.721256e+07  32.100898  0.000000e+00   
             GB       51.108000  2.282586e+07  40.680195  0.000000e+00   
             GR       51.596074  7.974345e+06  27.793720  0.000000e+00   
             HR       48.691380  1.479920e+05  32.717732  0.000000e+00   
             IT       52.272108  4.962200e+07  29.673773  0.000000e+00   
             LU       35.536662  2.066320e+05  40.884383  0.000000e+00   
             NL       39.087237  3.755508e+06  42.250375  0.000000e+00   
             PT       52.453883  1.724552e+06  28.678537  0.000000e+00   
             RO       47.900233  3.699221e+06  35.098223  0.000000e+00   
             SI       48.573128  5.673560e+05  36.692569  0.000000e+00   
             SK       36.349418  1.179207e+06  37.934628  0.000000e+00   
WindOffshore BE       43.510899  2.843207e+06  68.031302  0.000000e+00   
             DE       36.912644  1.794700e+07  58.281499  0.000000e+00   
             DK       35.633719  5.179772e+06  59.492800  0.000000e+00   
             GB       50.869553  2.244249e+07  60.065105  0.000000e+00   
             NL       39.427779  3.830780e+06  57.075555  0.000000e+00   
WindOnshore  AT       37.402832  6.727005e+06  21.922485  0.000000e+00   
             BE       43.192103  3.339857e+06  31.125242  0.000000e+00   
             BG       40.987382  1.447000e+06  29.245016  0.000000e+00   
             CH       42.962105  7.905059e+04  33.936992  6.424223e-06   
             CZ       36.698852  5.819025e+05  32.208293  0.000000e+00   
             DE       36.533432  8.865700e+07  29.905295  0.000000e+00   
             DK       35.510361  9.594894e+06  20.161813  0.000000e+00   
             ES       51.966647  4.783127e+07  25.692968  0.000000e+00   
             FI       38.289800  4.682626e+06  31.681138  0.000000e+00   
             FR       45.920469  2.284039e+07  28.034458  0.000000e+00   
             GB       50.880300  3.072025e+07  19.841961  0.000000e+00   
             GR       53.163973  5.529781e+06  19.298288  0.000000e+00   
             HR       48.507098  1.177879e+06  33.200512  0.000000e+00   
             HU       49.944114  7.371610e+05  29.810493  0.000000e+00   
             IE       43.078710  7.385241e+06  17.782109  0.000000e+00   
             IT       55.866466  1.750301e+07  21.916968  0.000000e+00   
             LU       37.042168  1.901360e+05  31.964476  2.404204e-06   
             NL       39.370353  7.312863e+06  24.092615  0.000000e+00   
             NO       36.041553  2.842768e+06  22.884622  0.000000e+00   
             PL       35.991571  1.453217e+07  32.664311  0.000000e+00   
             PT       51.578264  1.213530e+07  23.218076  0.000000e+00   
             RO       47.969995  7.335892e+06  27.002302  0.000000e+00   
             SE       36.035950  1.730753e+07  29.489367  0.000000e+00   
             SI       49.050627  5.710217e+03  37.944607  2.493887e-06   

                          a_new  
technology   country             
Solar        AT       36.276444  
             BE       41.509845  
             BG       40.832983  
             CH       42.004316  
             CZ       35.738482  
             DE       35.210593  
             DK       35.825999  
             ES  

## Set new parameters

Slope remains unaltered by intercept is increased to avoid investments

In [15]:
df_all["cinv_0"] = (df_all[["a", "a_new"]].max(1)).map(lambda x: ceil(x*10)/10)
df_all["cinv_1"] = df_all["b"]

In [16]:
df_all.head()

revenue    generation          a             b  \
technology country                                                     
Solar      AT       36.300167  1.144304e+06  36.213953  0.000000e+00   
           BE       41.545160  5.776014e+06  42.156420  0.000000e+00   
           BG       40.877746  2.652000e+06  32.704350  0.000000e+00   
           CH       42.018984  4.209379e+05  34.595313  0.000000e+00   
           CZ       35.824808  4.326927e+06  37.605184  0.000000e+00   

                        a_new  cinv_0        cinv_1  
technology country                                   
Solar      AT       36.276444    36.3  0.000000e+00  
           BE       41.509845    42.2  0.000000e+00  
           BG       40.832983    40.9  0.000000e+00  
           CH       42.004316    42.1  0.000000e+00  
           CZ       35.738482    37.7  0.000000e+00

## Export

In [17]:
df_all.to_excel(fn_out, sheet_name="to_gams", merge_cells=False)